# PIES Files

In [5]:
import xml.etree.ElementTree as ET
import pandas as pd
import numpy as np

In [ ]:
df_map=pd.read_excel(r"20260129_Autocare_PCAdb.xlsx")
df_map.PAID=df_map.PAID.astype(str)
df_map.PartTerminologyID=df_map.PartTerminologyID.astype(str)
df_map

In [ ]:
import xml.etree.ElementTree as ET
import pandas as pd

# Load the XML file (update the filename if needed)
xml_file = r"20260219_190932.xml"
tree = ET.parse(xml_file)
root = tree.getroot()

tree = ET.parse(xml_file)
root = tree.getroot()

# Define namespace if present (adjust if needed)
ns = {"ns": "http://www.autocare.org"}

# Find all Item elements
items = root.findall(".//ns:Item", ns)

extended_info_data = []
product_attr_data = []

# Loop through each Item element and extract the desired values.
for item in items:
    # Get the PartNumber and PartTerminologyID
    part_number_elem = item.find("ns:PartNumber", ns)
    part_term_elem = item.find("ns:PartTerminologyID", ns)
    part_number = part_number_elem.text if part_number_elem is not None else None
    part_terminology_id = part_term_elem.text if part_term_elem is not None else None

    # Extract ExtendedInformation values
    for ext in item.findall("ns:ExtendedInformation/ns:ExtendedProductInformation", ns):
        extended_info_data.append({
            "PartNumber": part_number,
            "PartTerminologyID": part_terminology_id,
            "EXPICode": ext.attrib.get("EXPICode"),
            "Value": ext.text
        })

    # Extract ProductAttributes values
    for attr in item.findall("ns:ProductAttributes/ns:ProductAttribute", ns):
        product_attr_data.append({
            "PartNumber": part_number,
            "PartTerminologyID": part_terminology_id,
            "AttributeID": attr.attrib.get("AttributeID"),
            "AttributeUOM": attr.attrib.get("AttributeUOM", ""),
            "Value": attr.text
        })

# Create DataFrames for the extracted data
df_extended = pd.DataFrame(extended_info_data)
df_product_attr = pd.DataFrame(product_attr_data)

df_product_attr

In [ ]:
import pandas as pd

# DataFrames with common 'Name' column
df1 = pd.DataFrame({'Name': ['John', 'Mary', 'Bob'], 'Age': [28, 25, 30]})
df2 = pd.DataFrame({'Name': ['Mary', 'Bob', 'John'], 'City': ['Chicago', 'LA', 'NY']})

# Use map to add the 'City' to df1
# We set the index of df2 to 'Name' for the mapping
df1['City'] = df1['Name'].map(df2.set_index('Name')['City'])
df1


In [16]:
df_product_attr["PartName"]=df_product_attr["PartTerminologyID"].map(
    df_map[["PartTerminologyID","PartTerminologyName"]].drop_duplicates()
    .set_index(["PartTerminologyID"])["PartTerminologyName"]
    )

df_product_attr["Attribute Name"]=df_product_attr["AttributeID"].map(
    df_map[["PAID","PAName"]].drop_duplicates()
    .set_index(["PAID"])["PAName"]
    )

df_product_attr['Attribute Name']=np.where(df_product_attr['Attribute Name'].isna(), df_product_attr['AttributeID'], df_product_attr['Attribute Name'])

df_product_attr=df_product_attr[["PartNumber","PartName","Attribute Name","Value","AttributeUOM"]]

In [ ]:
df_product_attr

In [18]:
len(df_product_attr['PartNumber'].unique().tolist())

465

In [ ]:
# Write the DataFrames to an Excel file with separate sheets
output_file = r"20260219_190932.xlsx"
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    #df_extended.to_excel(writer, sheet_name="ExtendedInformation", index=False)
    df_product_attr.to_excel(writer, sheet_name="ProductAttributes", index=False)

print(f"Data has been extracted and saved to {output_file}")


# ACES Files

## Batch File Extraction

In [ ]:
import os

path = r"Folder Path"
xml_files = []

for root, dirs, files in os.walk(path):
    for file in files:
        if file.endswith(".xml"):
            xml_files.append(os.path.join(root, file))

print("XML files found:")
for file in xml_files:
    print(file)


In [ ]:
for xml_file in xml_files:
    print(f"Processing file: {file}")
    # Add your XML processing code here
    tree = ET.parse(xml_file)
    root = tree.getroot()

    # ---- Extract Header Information ----
    header = root.find("Header")
    header_data = {child.tag: child.text for child in header if child.text}

    # Approved countries (multiple <Country> tags)
    countries = [c.text for c in header.findall("ApprovedFor/Country")]
    header_data["ApprovedCountries"] = ", ".join(countries)

    df_header = pd.DataFrame([header_data])
    # ---- Extract Applications ----
    apps = []
    for app in root.findall("App"):
        app_data = {
            # "AppID": app.attrib.get("id"),
            # "Action": app.attrib.get("action"),
            # "Validate": app.attrib.get("validate"),
            "BaseVehicle": app.findtext("BaseVehicle"),
            "BaseVehicleID": app.find("BaseVehicle").attrib.get("id"),
            "Qty": app.findtext("Qty"),
            "PartType": app.findtext("PartType"),
            # "PartTypeID": app.find("PartType").attrib.get("id"),
            "MfrLabel": app.findtext("MfrLabel"),
            "Position": app.findtext("Position"),
            # "PositionID": app.find("Position").attrib.get("id"),
            "Part": app.findtext("Part"),
            # "Aspiration":app.findtext("Aspiration"),
            # "EngineType":app.findtext("EngineType"),
            "MfrLabel":app.findtext("MfrLabel"),
            # "Note":app.findtext("Note"),
            # "CylinderHeadType":app.findtext("CylinderHeadType"),
            # "ValvesPerEngine":app.findtext("ValvesPerEngine"),
            # "EngineBase":app.findtext("EngineBase"),
            # "BodyType": app.findtext("BodyType"),
            # "DriveType": app.findtext("DriveType"),
            # "FuelType": app.findtext("FuelType"),
            # "Region": app.findtext("Region"),
            # "SubModel": app.findtext("SubModel"),
            # "BodyNumDoors": app.findtext("BodyNumDoors"),
            # "MfrBodyCode": app.findtext("MfrBodyCode"),
            # "FrontBrakeType": app.findtext("FrontBrakeType"),
            # "RearBrakeType": app.findtext("RearBrakeType"),
            # "BrakeSystem": app.findtext("BrakeSystem"),
            # "BrakeABS": app.findtext("BrakeABS"),
            # "FrontSpringType": app.findtext("FrontSpringType"),
            # "RearSpringType": app.findtext("RearSpringType"),
            # "SteeringType": app.findtext("SteeringType"),
            # "EngineDesignation": app.findtext("EngineDesignation"),
            
        }
        apps.append(app_data)
    df_apps = pd.DataFrame(apps)
    # df_apps[['Make', 'Model', 'Year']] = df_apps['BaseVehicle'].str.split(' - ', expand=True)
    df_apps[['Year', 'Make', 'Model']] = df_apps['BaseVehicle'].str.split(', ', expand=True)
    df_apps['Key']=df_apps['Part']+"$"+df_apps['PartType']+"$"+df_apps['Year']+"$"+df_apps['Make']+"$"+df_apps['Model']+"$"+df_apps['Position']
    # df_apps["EngineLiters"]=df_apps["EngineBase"].str.extract(r'([\d\.]+)L')
    # df_apps["EngineCC"]=df_apps["EngineBase"].str.extract(r'\((\d+)\)')
    # df_apps["EngineCylinder"]=df_apps["EngineBase"].str.extract(r'(\d+)Cyl')
    # df_apps["EngineBlock"]=df_apps["EngineBase"].str.extract(r'([A-Za-z]+) \(')
    # df_apps["EngBoreMetric"]=df_apps["EngineBase"].str.extract(r'(\d+\.\d+)\s*\(Bore\)')
    # df_apps["EngBoreInch"]=df_apps["EngineBase"].str.extract(r'(\d+\.\d+)\s+\d+(?:\.\d+)?\s*\(Bore\)')
    # df_apps["EngStrokeInch"]=df_apps["EngineBase"].str.extract(r'(\d+\.\d+)\s+\d+(?:\.\d+)?\s*\(Stroke\)')
    # df_apps["EngStrokeMetric"]=df_apps["EngineBase"].str.extract(r'\d+\.\d+\s+(\d+(?:\.\d+)?)\s*\(Stroke\)')
    del df_apps["BaseVehicle"]
    output_file=rf"FolderPath\{xml_file.split("\\")[-2]}_PartCat_Apps.csv"
    df_apps.to_csv(output_file, index=False, encoding="utf-8-sig")
    print(f"Data has been extracted and saved to {output_file}")

In [ ]:
df_apps

## Single File Extraction

In [ ]:
import xml.etree.ElementTree as ET
import pandas as pd

# Load the XML file (update the filename if needed)
xml_file = r"20251111_045439.xml"

tree = ET.parse(xml_file)
root = tree.getroot()

# ---- Extract Header Information ----
header = root.find("Header")
header_data = {child.tag: child.text for child in header if child.text}

# Approved countries (multiple <Country> tags)
countries = [c.text for c in header.findall("ApprovedFor/Country")]
header_data["ApprovedCountries"] = ", ".join(countries)

df_header = pd.DataFrame([header_data])


In [ ]:

# ---- Extract Applications ----
apps = []
for app in root.findall("App"):
    app_data = {
        "AppID": app.attrib.get("id"),
        "Action": app.attrib.get("action"),
        "Validate": app.attrib.get("validate"),
        "BaseVehicle": app.findtext("BaseVehicle"),
        "BaseVehicleID": app.find("BaseVehicle").attrib.get("id"),
        "Qty": app.findtext("Qty"),
        "PartType": app.findtext("PartType"),
        "PartTypeID": app.find("PartType").attrib.get("id"),
        "MfrLabel": app.findtext("MfrLabel"),
        "Position": app.findtext("Position"),
        "PositionID": app.find("Position").attrib.get("id"),
        "Part": app.findtext("Part"),
        "Aspiration":app.findtext("Aspiration"),
        "EngineType":app.findtext("EngineType"),
        "MfrLabel":app.findtext("MfrLabel"),
        "Note":app.findtext("Note"),
        "CylinderHeadType":app.findtext("CylinderHeadType"),
        "ValvesPerEngine":app.findtext("ValvesPerEngine"),
        "EngineBase":app.findtext("EngineBase"),
        "BodyType": app.findtext("BodyType"),
        "DriveType": app.findtext("DriveType"),
        "FuelType": app.findtext("FuelType"),
        "Region": app.findtext("Region"),
        "SubModel": app.findtext("SubModel"),
        "BodyNumDoors": app.findtext("BodyNumDoors"),
        "MfrBodyCode": app.findtext("MfrBodyCode"),
        "FrontBrakeType": app.findtext("FrontBrakeType"),
        "RearBrakeType": app.findtext("RearBrakeType"),
        "BrakeSystem": app.findtext("BrakeSystem"),
        "BrakeABS": app.findtext("BrakeABS"),
        "FrontSpringType": app.findtext("FrontSpringType"),
        "RearSpringType": app.findtext("RearSpringType"),
        "SteeringType": app.findtext("SteeringType"),
        "EngineDesignation": app.findtext("EngineDesignation"),
        
    }
    apps.append(app_data)
df_apps = pd.DataFrame(apps)


In [ ]:
df_apps[['Year', 'Make', 'Model']] = df_apps['BaseVehicle'].str.split(', ', expand=True)

In [ ]:
df_apps

In [5]:
df_apps["EngineLiters"]=df_apps["EngineBase"].str.extract(r'([\d\.]+)L')
df_apps["EngineCC"]=df_apps["EngineBase"].str.extract(r'\((\d+)\)')
df_apps["EngineCylinder"]=df_apps["EngineBase"].str.extract(r'(\d+)Cyl')
df_apps["EngineBlock"]=df_apps["EngineBase"].str.extract(r'([A-Za-z]+) \(')
df_apps["EngBoreMetric"]=df_apps["EngineBase"].str.extract(r'(\d+\.\d+)\s*\(Bore\)')
df_apps["EngBoreInch"]=df_apps["EngineBase"].str.extract(r'(\d+\.\d+)\s+\d+(?:\.\d+)?\s*\(Bore\)')
df_apps["EngStrokeInch"]=df_apps["EngineBase"].str.extract(r'(\d+\.\d+)\s+\d+(?:\.\d+)?\s*\(Stroke\)')
df_apps["EngStrokeMetric"]=df_apps["EngineBase"].str.extract(r'\d+\.\d+\s+(\d+(?:\.\d+)?)\s*\(Stroke\)')

In [ ]:
df_Filtered=pd.merge(df_apps,df_map[['Article Number','Parent article number']],left_on='Part',right_on='Article Number',how='inner')
df_Filtered

In [ ]:
#df_Filtered=df_Filtered[['Parent article number','Article Number','PartType',  'Part', 'Year','Make', 'Model', 'Position']]

In [ ]:
# ---- Save to Excel ----
output_file = r"ACES.xlsx"
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    df_Filtered.to_excel(writer, sheet_name="ACES", index=False)
print(f"Data has been extracted and saved to {output_file}")


# Misc Code

In [ ]:
import xml.etree.ElementTree as ET
import pandas as pd

# Load the XML file
xml_file = r"2025-09-01.xml"
tree = ET.parse(xml_file)
root = tree.getroot()

# Define namespaces (AutoCare PIES XML uses default namespace)
ns = {"ns": "http://www.autocare.org"}

# Extract Header Information
header = root.find("ns:Header", ns)
header_data = {child.tag.split("}")[1]: child.text for child in header}
df_header = pd.DataFrame([header_data])

# Extract Price Sheets
pricesheets = []
for ps in root.findall("ns:PriceSheets/ns:PriceSheet", ns):
    ps_data = {child.tag.split("}")[1]: child.text for child in ps}
    pricesheets.append(ps_data)
df_pricesheets = pd.DataFrame(pricesheets)

# Extract Items
items = []
extended_info = []
product_attributes = []

for item in root.findall("ns:Items/ns:Item", ns):
    part_number = item.find("ns:PartNumber", ns).text
    # brand_label = item.find("ns:BrandLabel", ns).text

    # Extract Item Data
    item_data = {
        "PartNumber": part_number,
        "BrandLabel": brand_label,
        # "GTIN": item.find("ns:ItemLevelGTIN", ns).text,
        # "MinimumOrderQuantity": item.find("ns:MinimumOrderQuantity", ns).text,
    }

    # Extract Descriptions
    descriptions = item.findall("ns:Descriptions/ns:Description", ns)
    for desc in descriptions:
        desc_code = desc.attrib.get("DescriptionCode", "Other")
        item_data[f"Description_{desc_code}"] = desc.text

    # Extract Prices
    prices = item.findall("ns:Prices/ns:Pricing", ns)
    for price in prices:
        price_type = price.attrib.get("PriceType", "Other")
        item_data[f"Price_{price_type}"] = price.find("ns:Price", ns).text

    items.append(item_data)

    # Extract Extended Information
    for ext in item.findall("ns:ExtendedInformation/ns:ExtendedProductInformation", ns):
        extended_info.append({
            "PartNumber": part_number,
            "EXPICode": ext.attrib.get("EXPICode"),
            "Value": ext.text
        })

    # Extract Product Attributes
    for attr in item.findall("ns:ProductAttributes/ns:ProductAttribute", ns):
        product_attributes.append({
            "PartNumber": part_number,
            "AttributeID": attr.attrib.get("AttributeID"),
            "Value": attr.text
        })

df_items = pd.DataFrame(items)
df_extended_info = pd.DataFrame(extended_info)
df_product_attributes = pd.DataFrame(product_attributes)

# Extract Marketing Copy
marketing_copy = []
for mc in root.findall("ns:MarketingCopy/ns:MarketCopy/ns:MarketCopyContent", ns):
    mc_data = {
        "MarketCopyCode": mc.attrib.get("MarketCopyCode"),
        "MarketCopyType": mc.attrib.get("MarketCopyType"),
        "Content": mc.text,
    }
    marketing_copy.append(mc_data)

df_marketing_copy = pd.DataFrame(marketing_copy)



# Save to Excel with different sheets
output_file = r"Outputfilepath/file.xlsx"
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    df_header.to_excel(writer, sheet_name="Header", index=False)
    df_pricesheets.to_excel(writer, sheet_name="PriceSheets", index=False)
    df_items.to_excel(writer, sheet_name="Items", index=False)
    df_extended_info.to_excel(writer, sheet_name="ExtendedInformation", index=False)
    df_product_attributes.to_excel(writer, sheet_name="ProductAttributes", index=False)
    df_marketing_copy.to_excel(writer, sheet_name="MarketingCopy", index=False)

print(f"Data has been extracted and saved to {output_file}")

